# GROTH-07b : les bornes de croissance — compagnon natif

**Navigation** : [Index de la série](../README.md) · [Îlot Grothendieck](README.md) · [Carnet Python](GROTH-07-Capacity-Python.ipynb) | [< GROTH-06 Groupes formels](GROTH-06-FormalGroups-Lean.ipynb)

**Kernel** : Lean 4 (WSL) — lac [`gkdim_lean/`](gkdim_lean/), mathlib au pin `db584cd6`

***

Ce carnet rend visibles, **sous le vrai kernel Lean 4**, les lemmes de croissance du
lac `gkdim_lean`. « Natif » signifie ici que les imports, les types et les
certificats sont contrôlés par le compilateur : la prose explique les sorties, mais ne
s'y substitue pas.

Le lac expose deux bornes de croissance — une borne **supérieure** (`growthAtMost`) et
une borne **inférieure** (`growthAtLeast`) — et montre que la fonction de Hilbert de
l'anneau polynomial à une variable les satisfait toutes deux. C'est la forme jouable
de la définition de la dimension de Gelfand-Kirillov : un exposant est exactement la
grandeur caractérisée par ces deux bornes.

**Ce que ce carnet ne fait pas.** Il ne formalise pas le théorème du trou de
Smoktunowicz : la bibliothèque de maths de Lean ne sait ni définir la dimension de
Gelfand-Kirillov, ni Graduer un anneau par `N`, ni localiser d'Ore. Le lac se place
**en deçà** de cette frontière — il fournit le premier jalon.

## 1. Le module et ses déclarations

Le lac `gkdim_lean/` est un projet Lake minimal : un `lakefile.lean`, un
`lean-toolchain` épinglé, un `lake-manifest.json` épinglant Mathlib, et un module
`GKdim` décliné en jumeau français / anglais (`GKdim/Core.lean` et
`GKdim/Core_en.lean`, namespaces `GKdim` et `GKdim_en`).

On importe le module et on interroge le noyau sur ses déclarations.


In [1]:
import GKdim.Core

open GKdim

#check GKdim.hilbert_poly1
#check GKdim.growthAtMost
#check GKdim.growthAtLeast
#check GKdim.poly1_growthAtMost
#check GKdim.poly1_growthAtLeast
#check GKdim.poly1_gkdim_is_one


import GKdim.Core

open GKdim

#check GKdim.hilbert_poly1
──────▶  GKdim.hilbert_poly1 (n : ℕ) : ℕ
#check GKdim.growthAtMost
──────▶  GKdim.growthAtMost (G : ℕ → ℕ) (d : ℕ) : Prop
#check GKdim.growthAtLeast
──────▶  GKdim.growthAtLeast (G : ℕ → ℕ) (d : ℕ) : Prop
#check GKdim.poly1_growthAtMost
──────▶  GKdim.poly1_growthAtMost : growthAtMost hilbert_poly1 1
#check GKdim.poly1_growthAtLeast
──────▶  GKdim.poly1_growthAtLeast : growthAtLeast hilbert_poly1 1
#check GKdim.poly1_gkdim_is_one
──────▶  GKdim.poly1_gkdim_is_one : growthAtMost hilbert_poly1 1 ∧ growthAtLeast hilbert_poly1 1

--% env 0

Raw input:
{"cmd": "import GKdim.Core\n\nopen GKdim\n\n#check GKdim.hilbert_poly1\n#check GKdim.growthAtMost\n#check GKdim.growthAtLeast\n#check GKdim.poly1_growthAtMost\n#check GKdim.poly1_growthAtLeast\n#check GKdim.poly1_gkdim_is_one\n"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data": "GKdim.hilbert_poly1 (n : ℕ) : ℕ"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "GKdim.growthAtMost (G : ℕ → ℕ) (d : ℕ) : Prop"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data": "GKdim.growthAtLeast (G : ℕ → ℕ) (d : ℕ) : Prop"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data": "GKdim.poly1_growthAtMost : growthAtMost hilbert_poly1 1"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data": "GKdim.poly1_growthAtLeast : growthAtLeast hilbert_poly1 1"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "GKdim.poly1_gkdim_is_one : growthAtMost hilbert_poly1 1 ∧ growthAtLeast hilbert_poly1 1"}],
 "env": 0}

### Lecture du résultat : deux bornes, une croissance

Le noyau confirme la forme des deux bornes :

- `growthAtMost G d` se lit « il existe une constante `C` et un seuil `n₀` tels que
  `G n ≤ C * n ^ d` pour tout `n ≥ n₀` » ;
- `growthAtLeast G d` se lit « il existe `c > 0` et un seuil `n₀` tels que
  `G n ≥ c * n ^ d` pour tout `n ≥ n₀` ».

Les deux bornes sont ce qui distingue un exposant **exactement** égal à `d` d'un
exposant seulement majoré ou seulement minoré : c'est leur réunion qui fixe
l'exposant, et c'est donc elle qui définit la dimension de Gelfand-Kirillov.

## 2. Les certificats d'axiomes

Un théorème Lean qui ne repose que sur des axiomes standards se prouve sans
`Classical.choice` et sans `sorryAx` : `#print axioms` liste exactement les
hypothèses mobilisées. On l'applique aux trois lemmes du lac.


In [2]:
#print axioms GKdim.poly1_growthAtMost
#print axioms GKdim.poly1_growthAtLeast
#print axioms GKdim.poly1_gkdim_is_one


#print axioms GKdim.poly1_growthAtMost
──────▶  'GKdim.poly1_growthAtMost' depends on axioms: [propext, Quot.sound]
#print axioms GKdim.poly1_growthAtLeast
──────▶  'GKdim.poly1_growthAtLeast' depends on axioms: [propext, Quot.sound]
#print axioms GKdim.poly1_gkdim_is_one
──────▶  'GKdim.poly1_gkdim_is_one' depends on axioms: [propext, Quot.sound]

--% env 1

Raw input:
{"cmd": "#print axioms GKdim.poly1_growthAtMost\n#print axioms GKdim.poly1_growthAtLeast\n#print axioms GKdim.poly1_gkdim_is_one\n", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'GKdim.poly1_growthAtMost' depends on axioms: [propext, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'GKdim.poly1_growthAtLeast' depends on axioms: [propext, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'GKdim.poly1_gkdim_is_one' depends on axioms: [propext, Quot.sound]"}],
 "env": 1}

### Lecture du résultat : des lemmes sans hypothèse classique

Les trois certificats sont identiques et ne citent que `propext` et `Quot.sound` — les
deux axiomes de base de Lean 4, déjà combinés dans les lemmes de Mathlib. Ni
`Classical.choice`, ni `sorryAx` : les lemmes ne mobilisent aucune hypothèse
classique et ne laissent aucun trou. Ce sont des lemmes vérifiés par le noyau, pas
des énoncés assertés.

C'est la garantie formelle qui rend le carnet Python vérifiable : les bornes que la
section 1 y mesure numériquement sont ici énoncées et prouvées.

## 3. Les bornes, prouvées à la main

On rejoue les deux bornes comme des `example` indépendants, pour montrer que la
démonstration du lac se réduit à de l'arithmétique sur les naturels.


In [3]:
-- Borne supérieure : la fonction de Hilbert de K[X] est majorée par 2 * n.
example (n : ℕ) (hn : 1 ≤ n) : GKdim.hilbert_poly1 n ≤ 2 * n := by
  simp only [GKdim.hilbert_poly1]
  omega

-- Borne inférieure : elle est minorée par n (le terme dominant).
example (n : ℕ) (hn : 1 ≤ n) : GKdim.hilbert_poly1 n ≥ n := by
  simp only [GKdim.hilbert_poly1]
  omega


-- Borne supérieure : la fonction de Hilbert de K[X] est majorée par 2 * n.
example (n : ℕ) (hn : 1 ≤ n) : GKdim.hilbert_poly1 n ≤ 2 * n := by
  simp only [GKdim.hilbert_poly1]
  omega

-- Borne inférieure : elle est minorée par n (le terme dominant).
example (n : ℕ) (hn : 1 ≤ n) : GKdim.hilbert_poly1 n ≥ n := by
                 ──▶ 🟨 Variable name `hn` is not explicitly referenced.

Hint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:
  [apply] _hn

Note: This linter can be disabled with `set_option linter.unusedVariables false`
  simp only [GKdim.hilbert_poly1]
  omega

--% env 2

Raw input:
{"cmd": "-- Borne sup\u00e9rieure : la fonction de Hilbert de K[X] est major\u00e9e par 2 * n.\nexample (n : \u2115) (hn : 1 \u2264 n) : GKdim.hilbert_poly1 n \u2264 2 * n := by\n  simp only [GKdim.hilbert_poly1]\n  omega\n\n-- Borne inf\u00e9rieure : elle est minor\u00e9e par n (le terme dominant).\nexample (n : \u2115) (hn : 1 \u2264 n) : GKdim.hilbert_poly1 n \u2265 n := by\n  simp only [GKdim.hilbert_poly1]\n  omega\n", "env": 1}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 7, "column": 17},
   "endPos": {"line": 7, "column": 19},
   "data":
   "Variable name `hn` is not explicitly referenced.\n\nHint: The binding can be removed (if unused) or named `_` (if used implicitly). Alternatively, prefix the name with `_` to silence this warning:\n  [apply] _hn\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"}],
 "env": 2}

### Lecture du résultat : la croissance est exactement linéaire

Les deux `example` compilent sans erreur : le noyau accepte `hilbert_poly1 n ≤ 2 * n`
et `hilbert_poly1 n ≥ n` pour tout `n ≥ 1`. La croissance de `K[X]` est donc
exactement linéaire, ce qui fixe la dimension de Gelfand-Kirillov de `K[X]` à `1` — le
théorème `poly1_gkdim_is_one` du lac.

C'est le pendant formel de la mesure numérique du carnet Python : là-bas on lit un
exposant sur une pente log-log, ici on lit un exposant sur la réunion de deux bornes.

## 4. Quelques valeurs

Le noyau évalue la fonction de Hilbert sur les premiers degrés.


In [4]:
#eval GKdim.hilbert_poly1 0
#eval GKdim.hilbert_poly1 3
#eval GKdim.hilbert_poly1 10


#eval GKdim.hilbert_poly1 0
─────▶  1
#eval GKdim.hilbert_poly1 3
─────▶  4
#eval GKdim.hilbert_poly1 10
─────▶  11

--% env 3

Raw input:
{"cmd": "#eval GKdim.hilbert_poly1 0\n#eval GKdim.hilbert_poly1 3\n#eval GKdim.hilbert_poly1 10\n", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 5},
   "data": "1"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "4"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "11"}],
 "env": 3}

### Lecture du résultat : la suite des dimensions

Les trois évaluations donnent les dimensions de l'espace des polynômes de degré au
plus `n` en une variable : une pour le degré nul, quatre au degré trois, onze au
degré dix. La suite croît d'une unité par degré, et c'est cette croissance
exactement linéaire que les deux bornes caractérisent.

## Conclusion

Le noyau Lean a chargé le module `GKdim`, vérifié les types des deux bornes de
croissance, délivré trois certificats d'axiomes réduits à `propext` et `Quot.sound`,
accepté deux `example` qui rejouent les bornes à la main, et évalué la fonction de
Hilbert sur quelques degrés.

Ce carnet ne franchit pas la frontière de la bibliothèque de maths de Lean : il pose
le premier jalon formel du parcours — la croissance linéaire de l'anneau polynomial à
une variable, donc une dimension de Gelfand-Kirillov égale à `1`. La frontière elle-même
(définir `GKdim`, graduer un anneau par `N`, localiser d'Ore) reste entière, et c'est
là que se situe le théorème du trou.

**Navigation** : [<< GROTH-06 Groupes formels](GROTH-06-FormalGroups-Lean.ipynb) |
[Carnet Python](GROTH-07-Capacity-Python.ipynb) |
[Îlot Grothendieck](README.md) | [Index de la série](../README.md)